# Analysis Notebook for Keithley CLI data

This notebook is used for analyzing the measurement data obtained from the Keithley CLI. It includes functions to download and combine `.txt` files from a Nextcloud share into a single DataFrame, as well as cells showing how to visualize the data using Plotly Express.

In [10]:
import plotly.express as px
import pandas as pd
from helpers import nextcloud_txt_to_dataframe, nextcloud_csv_to_dataframe

In [11]:
## Download and combine .txt files into a DataFrame, then print row counts per source file.
df = nextcloud_txt_to_dataframe()
df_joshua = nextcloud_csv_to_dataframe()

print("RADEF txt files:")
for file in df["source_file"].unique() if not df.empty else []:
    subset = df[df["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

print("Joshua's CSV files:")
for file in df_joshua["source_file"].unique() if not df_joshua.empty else []:
    subset = df_joshua[df_joshua["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

if df.empty:
    print("No data loaded from the Nextcloud share.")


RADEF txt files:
File: raw_data/dev3_idvg.txt, Data Rows: 51
File: raw_data/gq_test2_short_cable_1nA_gq.txt, Data Rows: 1966
File: raw_data/gq_test3_short_cable_1nA_1V_gq.txt, Data Rows: 1276
File: raw_data/gq_test3_short_cable_1nA_gq.txt, Data Rows: 1385
File: raw_data/gq_test5_gq.txt, Data Rows: 2684
File: raw_data/gq_test6_gq.txt, Data Rows: 145
File: raw_data/gq_test_100nA_gq.txt, Data Rows: 14
File: raw_data/gq_test_20nA_gq.txt, Data Rows: 72
File: raw_data/gq_test_dev2_1nA_gq.txt, Data Rows: 691
File: raw_data/gq_test_dev2_20nA_gq.txt, Data Rows: 481
File: raw_data/gq_test_dev2_2nA_gq.txt, Data Rows: 700
File: raw_data/gq_test_short_cable_1nA_1V_cable_test2_gq.txt, Data Rows: 1384
File: raw_data/gq_test_short_cable_1nA_1V_cable_test3_gq.txt, Data Rows: 1386
File: raw_data/gq_test_short_cable_1nA_1V_cable_test_gq.txt, Data Rows: 2643
File: raw_data/gq_test_short_cable_1nA_1V_dev3_gq.txt, Data Rows: 1363
File: raw_data/gq_test_short_cable_1nA_gq.txt, Data Rows: 489
File: raw_data/g

## Gate Charge Data

In [12]:
# Get gate charge data and plot gate voltage vs. gate charge for files containing "_gq" in their name.

gate_charge_df = df[df["source_file"].str.contains("_gq", case=False, na=False)]

if gate_charge_df.empty:
    print("No gate charge data found in the DataFrame.")
else:
    gate_charge_df["Gate Charge"] = (
        gate_charge_df["Gate Current"] * gate_charge_df["Time"]
    )

    fig = px.line(
        gate_charge_df,
        x="Gate Charge",
        y="Gate Voltage",
        color="source_file",
        title="Gate Voltage vs. Gate Charge",
    )

    fig.update_layout(
        xaxis_title="Gate Charge (Coulombs)",
        yaxis_title="Gate Voltage (Volts)",
        legend_title="Source File",
        width=800,
        height=600,
        xaxis_range=[0, 50e-9],
    )
    fig.show()

## ID-VD Data

In [13]:
# Get ID-VD data and plot Drain Current and Gate Current vs. Drain Voltage on the same graph, with separate lines for each source file.
idvd_df = df[df["source_file"].str.contains("_idvd", case=False, na=False)]
if idvd_df.empty:
    print("No ID-VD data found in the DataFrame.")
else:
    fig_idvd = px.line(
        idvd_df,
        x="Drain Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        title="ID-VD Characteristics",
        labels={"value": "Current (Amps)", "variable": "Current Type"},
    )
    fig_idvd.update_layout(
        xaxis_title="Drain Voltage (Volts)",
        yaxis_title="Current (Amps)",
        legend_title="Source File",
        width=800,
        height=600,
    )
    fig_idvd.show()

No ID-VD data found in the DataFrame.


## ID-VG Data

In [14]:
# Get ID-VG data and plot Drain Current and Gate Current vs. Gate Voltage on the same graph, with separate lines for each source file.
idvg_df = df[df["source_file"].str.contains("_idvg", case=False, na=False)]
idvg_df_joshua = df_joshua[
    df_joshua["source_file"].str.contains("_idvg", case=False, na=False)
]
# Remove DataName column if it exists, as it may interfere with plotting
idvg_df_joshua = idvg_df_joshua.drop(columns=["DataName"], errors="ignore")

# Rename joshua's ID-VG columns to match RADEF's for easier plotting
col_mapping = {
    "Vgate": "Gate Voltage",
    "Isource": "Source Voltage",
    "Idrain": "Drain Current",
    "Igate": "Gate Current",
}

# strip whitespace from column names in joshua's DataFrame before renaming
idvg_df_joshua.columns = idvg_df_joshua.columns.str.strip()

idvg_df_joshua = idvg_df_joshua.rename(columns=col_mapping)

idvg_df_joshua

,Gate Voltage,Source Voltage,Drain Current,Gate Current,source_file
0,-10,2.8905999999999997E-07,-5.5703E-07,-2.9850000000000003E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
1,-9.84,2.8494E-07,-3.6840000000000004E-08,-2.54E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
2,-9.68,2.789E-07,6.1542000000000009E-07,-2.585E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
3,-9.52,2.7268E-07,-8.054E-08,-2.49E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
4,-9.36,2.707E-07,3.0857E-07,-2.55E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
...,...,...,...,...,...
121,9.36,-0.008002,0.00799767,2.785E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
122,9.52,-0.0080016,0.00800604,2.84E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
123,9.68,-0.0080018,0.00799067,2.84E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv
124,9.84,-0.008002,0.00799744,3.015E-07,raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv


In [15]:
# Combine both dataframes
idvg_combined = pd.concat([idvg_df, idvg_df_joshua], ignore_index=True)

if idvg_combined.empty:
    print("No ID-VG data found in the DataFrame.")
else:
    fig_idvg = px.line(
        idvg_combined,
        x="Gate Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        title="ID-VG Characteristics",
    )

    fig_idvg.update_layout(
        xaxis_title="Gate Voltage (Volts)",
        yaxis_title="Current (Amps)",
        legend_title="Source File",
        width=800,
        height=600,
        yaxis_type="linear",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvg.show()

## Irradiation Data

In [16]:
## Irradiation Data
irrad_df = df[df["source_file"].str.contains("_irradiation", case=False)]
if irrad_df.empty:
    print("No irradiation data found in the DataFrame.")
else:
    fig_irrad = px.line(
        irrad_df,
        x="Time",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        title="Irradiation Test Data",
    )

    fig_irrad.update_layout(
        xaxis_title="Time (s)",
        yaxis_title="Current (Amps)",
        legend_title="Source File",
        width=800,
        height=600,
    )
    fig_irrad.show()

No irradiation data found in the DataFrame.
